# Phase 1: Business Understanding
**CRISP-DM Purpose:** Define the business problem, objectives, and success criteria before any data work begins.

---

## Problem Statement

| | |
|---|---|
| **Business Question** | Which active donors are most likely to go inactive within the next 90 days, so that board members and admins can prioritize personal outreach before the relationship lapses? |
| **Target Variable** | `at_risk` (engineered) |
| **Problem Type** | Binary Classification |
| **Positive Class** | `1 = at risk of going inactive` |
| **Prediction Horizon** | 90 days |
| **Output** | Risk score (0–1 probability) per active donor, written to `donor_risk_scores` in Supabase |
| **Consumer** | Board members and admins — review a prioritized list and conduct personal outreach calls |

---

## Feasibility Assessment

| Criterion | Assessment |
|---|---|
| **Practical Impact** | Retaining a lapsing donor is significantly cheaper than acquiring a new one. Early identification enables personal outreach before the relationship is lost. |
| **Data Availability** | Two tables available: `supporters` (demographics, acquisition channel, status) and `donations` (history, type, recency, amount). Both are in Supabase and exportable as CSV. |
| **Analytical Feasibility** | Sufficient features exist to build behavioral profiles (recency, frequency, monetary value, donor type, channel). Dataset is small (~60 donors) — model must be regularized; results should be interpreted with caution until data grows. |
| **Label Feasibility** | `at_risk` label is engineered from donation recency (no donation in 90 days). This is a reasonable and defensible proxy for inactivity risk. |

---

## Success Criteria

| | |
|---|---|
| **Primary metric** | Recall (at-risk class) — minimize missed at-risk donors |
| **Secondary metric** | ROC AUC — overall discrimination ability |
| **Baseline to beat** | Majority-class accuracy (computed below) |
| **Minimum acceptable Recall** | ≥ 0.70 — catch at least 70% of donors who will go inactive |
| **Minimum acceptable ROC AUC** | ≥ 0.75 — meaningfully better than random (0.50) |

> **Why Recall over Accuracy?**  
> The cost of a false negative (missing an at-risk donor, losing them silently) significantly outweighs the cost of a false positive (an unnecessary outreach call to a loyal donor). Therefore we optimize to minimize missed at-risk cases.

---

## Error Cost Analysis

| Error Type | Scenario | Estimated Cost |
|---|---|---|
| **False Negative** | Model predicts donor is safe → no outreach → donor goes inactive | High — lost donor relationship, lost future donations |
| **False Positive** | Model flags a loyal donor as at-risk → unnecessary outreach call | Low — a donor receiving an unexpected personal call is unlikely to be harmed; may even strengthen relationship |

**Implication for metric choice:** Prioritize Recall. Accept a higher false positive rate to ensure at-risk donors are not missed. Threshold on `risk_score` can be tuned below 0.5 to further increase recall at the cost of precision.

---

## Stakeholder Impact

| | |
|---|---|
| **Primary users** | Board members and admins |
| **Workflow** | Weekly: run inference → review prioritized risk list → assign personal outreach calls |
| **Decisions automated** | None — model surfaces risk, humans decide who to call |
| **Human-in-the-loop points** | All outreach decisions remain with board/admins |
| **Expected business value** | Retain at-risk donors before lapse; reduce donor churn rate; protect recurring donation revenue |
| **Prediction sink** | `donor_risk_scores` table in Supabase, keyed by `supporter_id` |

---

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Allow imports from src/
sys.path.insert(0, str(Path('..').resolve()))

from src.config import AT_RISK_DAYS

# Load raw data — check both possible locations
raw_dir = Path('..') / 'data' / 'raw'
fallback_dir = Path('..').parent  # pipeline/ folder where CSVs currently live

def find_csv(name):
    p = raw_dir / name
    if p.exists():
        return p
    return fallback_dir / name

supporters = pd.read_csv(find_csv('../../datasets/supporters.csv'), parse_dates=['created_at', 'first_donation_date'])
donations  = pd.read_csv(find_csv('../../datasets/donations.csv'),  parse_dates=['donation_date'])

print(f'Supporters: {len(supporters)} rows')
print(f'Donations:  {len(donations)} rows')
print(f'\nSupporter status distribution:')
print(supporters['status'].value_counts())

Supporters: 60 rows
Donations:  420 rows

Supporter status distribution:
status
Active      45
Inactive    15
Name: count, dtype: int64


In [3]:
from src.features import build_donation_features, add_label

df = build_donation_features(supporters, donations)
df = add_label(df, at_risk_days=AT_RISK_DAYS)

# Focus on active donors only (these are who we score in inference)
active = df[df['status'] == 'Active']

label_counts = active['at_risk'].value_counts()
majority_baseline = label_counts.max() / label_counts.sum()

print(f'Active donors: {len(active)}')
print(f'\nat_risk label distribution (among active donors):')
print(label_counts.rename({0: 'Not at risk (0)', 1: 'At risk (1))'}))
print(f'\nClass imbalance ratio: {label_counts.max() / label_counts.min():.1f}:1')
print(f'Majority-class baseline accuracy: {majority_baseline:.1%}')
print(f'\n→ A model must beat {majority_baseline:.1%} accuracy AND achieve Recall ≥ 0.70 to be useful.')

Active donors: 45

at_risk label distribution (among active donors):
at_risk
At risk (1))       34
Not at risk (0)    11
Name: count, dtype: int64

Class imbalance ratio: 3.1:1
Majority-class baseline accuracy: 75.6%

→ A model must beat 75.6% accuracy AND achieve Recall ≥ 0.70 to be useful.


## Phase Evidence & Assumptions

| Decision | Source | Notes |
|---|---|---|
| `at_risk` = no donation in 90 days | User-confirmed | Proxy label; may be revisited if historical snapshot data becomes available |
| Recall as primary metric | User-confirmed (false negatives are more costly) | Threshold may be tuned below 0.5 in Phase 5 |
| Train only on active donors | Design decision | Inactive donors have already lapsed — we want to learn patterns from donors who are currently in play |
| `days_since_last_donation` excluded as feature | Anti-leakage rule | This feature directly encodes the label; including it would make the model trivially accurate but useless for prediction |
| Dataset is small (~60 donors) | Observed | Models will be regularized; performance estimates have wide confidence intervals until dataset grows |

---

## Phase 1 Conclusion

**Problem is well-defined and feasible.**

We are building a binary classifier to score active donors by their probability of going inactive within 90 days. The output is a weekly risk list reviewed by board members and admins who conduct personal outreach calls. Success is defined as Recall ≥ 0.70 and ROC AUC ≥ 0.75 on held-out test data.

The key risk is dataset size (~60 donors). Results are directionally useful now but will improve significantly as more donors are added to the system.

---
**Proceed to Phase 2: Data Understanding →** `02_data_understanding.ipynb`